In [8]:
from pyspark import SparkConf, SparkContext
sc = SparkContext.getOrCreate()

# View spark context details (gives you Spark UI URL)
# Usually: http://master:4040
sc

<SparkContext master=local[*] appName=My App>

In [9]:
# Create array and RDD
x = [1,2,3,4,5,6,7,8,9,10,11,12]
xRDD = sc.parallelize(x)

# Question 2 Answer
print("Type of x:", type(x))
print("Type of xRDD:", type(xRDD))
print("API Documentation URL: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html")

Type of x: <class 'list'>
Type of xRDD: <class 'pyspark.rdd.RDD'>
API Documentation URL: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html


In [16]:
# Q3 - glom() function
print("Q3 - What glom() does:")
print("glom() coalesces all elements within each partition into a list")
print("Returns an RDD where each element is a list of partition contents")
print("")
print("Example:")
print("Before glom:", xRDD.collect())
print("After glom:", xRDD.glom().collect())

Q3 - What glom() does:
glom() coalesces all elements within each partition into a list
Returns an RDD where each element is a list of partition contents

Example:
Before glom: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
After glom: [[1], [2, 3], [4], [5, 6], [7], [8, 9], [10], [11, 12]]


In [17]:
# Q4 - Three print differences
print("=== 1. print(xRDD) ===")
print(xRDD)
print("→ Shows only RDD object description, NOT actual data")

print("\n=== 2. print(xRDD.collect()) ===")
print(xRDD.collect())
print("→ Brings ALL data to driver as one flat list")

print("\n=== 3. print(xRDD.glom().collect()) ===")
print(xRDD.glom().collect())
print("→ Shows data grouped by partition, one list per partition")

=== 1. print(xRDD) ===
ParallelCollectionRDD[1] at readRDDFromFile at PythonRDD.scala:289
→ Shows only RDD object description, NOT actual data

=== 2. print(xRDD.collect()) ===
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
→ Brings ALL data to driver as one flat list

=== 3. print(xRDD.glom().collect()) ===
[[1], [2, 3], [4], [5, 6], [7], [8, 9], [10], [11, 12]]
→ Shows data grouped by partition, one list per partition


In [18]:
# Q5 - Array of 50 default partitions
x50 = list(range(1, 51))
xRDD50 = sc.parallelize(x50)

print("Array size:", len(x50))
print("Default partitions:", xRDD50.getNumPartitions())
print("Partition contents:", xRDD50.glom().collect())
print("")
print("Explanation:")
print("Default partitions =", xRDD50.getNumPartitions())
print("This equals number of CPU cores (local[*] uses all cores)")

Array size: 50
Default partitions: 8


[Stage 10:====================================>                     (5 + 3) / 8]

Partition contents: [[1, 2, 3, 4, 5, 6], [7, 8, 9, 10, 11, 12], [13, 14, 15, 16, 17, 18], [19, 20, 21, 22, 23, 24], [25, 26, 27, 28, 29, 30], [31, 32, 33, 34, 35, 36], [37, 38, 39, 40, 41, 42], [43, 44, 45, 46, 47, 48, 49, 50]]

Explanation:
Default partitions = 8
This equals number of CPU cores (local[*] uses all cores)


In [20]:
# Q6 - Repartition to 7
xRDD50_7 = xRDD50.repartition(7)

print("Partitions after repartition:", xRDD50_7.getNumPartitions())
contents = xRDD50_7.glom().collect()
print("Output of the Collect\n",contents)
print("After Arrange")

for i, partition in enumerate(contents):
    print(f"Partition {i}: {len(partition)} elements → {partition}")

print("")
sizes = xRDD50_7.glom().map(len).collect()
print("Workload per thread:", sizes)
print("Total elements:", sum(sizes))

Partitions after repartition: 7


Output of the Collect
 [[19, 20, 21, 22, 23, 24], [], [13, 14, 15, 16, 17, 18, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 43, 44, 45, 46, 47, 48, 49, 50], [1, 2, 3, 4, 5, 6], [37, 38, 39, 40, 41, 42], [], [7, 8, 9, 10, 11, 12]]
After Arrange
Partition 0: 6 elements → [19, 20, 21, 22, 23, 24]
Partition 1: 0 elements → []
Partition 2: 26 elements → [13, 14, 15, 16, 17, 18, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 43, 44, 45, 46, 47, 48, 49, 50]
Partition 3: 6 elements → [1, 2, 3, 4, 5, 6]
Partition 4: 6 elements → [37, 38, 39, 40, 41, 42]
Partition 5: 0 elements → []
Partition 6: 6 elements → [7, 8, 9, 10, 11, 12]

Workload per thread: [6, 0, 26, 6, 6, 0, 6]
Total elements: 50


In [23]:
# Q7 - Execution order
print("Observing execution order:")

def f(iterator):
    import os
    pid = os.getpid()
    items = list(iterator)
    if items:
        print(f"Process {pid} handled: {items}")
    yield None

xRDD.foreachPartition(f)

print("")
print("Q7 Answer:")
print("Partitions do NOT execute in sequential order (0,1,2...)")
print("Spark schedules partitions based on available cores")
print("Order is non-deterministic and depends on resource availability")

Observing execution order:

Q7 Answer:
Partitions do NOT execute in sequential order (0,1,2...)
Spark schedules partitions based on available cores
Order is non-deterministic and depends on resource availability


Process 22802 handled: [1]                                          (0 + 8) / 8]
Process 22789 handled: [4]
Process 22782 handled: [5, 6]
Process 22993 handled: [2, 3]
Process 22989 handled: [8, 9]
Process 22795 handled: [7]
Process 22995 handled: [10]
Process 22785 handled: [11, 12]
                                                                                

In [24]:
# Q8 - Most time consuming job from Spark UI
print("Q8 Instructions:")
print("1. Open browser: http://hadoop-master:4040")
print("2. Click 'Jobs' tab")
print("3. Sort by Duration column")
print("4. Find job with LONGEST duration")
print("")
print("Record from Spark UI:")
print("Job ID          = [check UI]")
print("Job Description = [check UI]")
print("Duration        = [check UI] seconds")

# Run something to make it appear in UI
result = xRDD50.reduce(lambda a,b: a+b)
print("\nSum of 1-50:", result)
print("Now check Spark UI for this job!")

Q8 Instructions:
1. Open browser: http://hadoop-master:4040
2. Click 'Jobs' tab
3. Sort by Duration column
4. Find job with LONGEST duration

Record from Spark UI:
Job ID          = [check UI]
Job Description = [check UI]
Duration        = [check UI] seconds


[Stage 23:=======>                                                  (1 + 7) / 8]


Sum of 1-50: 1275
Now check Spark UI for this job!


In [25]:
# Q9 - foreachPartition DAG
print("Running foreachPartition for DAG...")

def f(iterator):
    for x in iterator:
        print(x)
    yield None

xRDD.foreachPartition(f)




Running foreachPartition for DAG...


8Stage 24:>                                                         (0 + 8) / 8]
9
5
6
42
3

10
11
12
7
1
                                                                                

In [27]:
xRDDEven = xRDD.filter(lambda y: y % 2 == 0)
xRDDOdd  = xRDD.filter(lambda y: y % 2 == 1)
xRDDUnion = xRDDEven.union(xRDDOdd)

print("Even:", xRDDEven.collect())
print("Odd:", xRDDOdd.collect())
print("Union:", xRDDUnion.collect())
print("Union partitions:", xRDDUnion.getNumPartitions())

Even: [2, 4, 6, 8, 10, 12]
Odd: [1, 3, 5, 7, 9, 11]
Union: [2, 4, 6, 8, 10, 12, 1, 3, 5, 7, 9, 11]
Union partitions: 16
